# Week 8 — Evaluation Expansion

Loads the tuned gradient boosting model and test predictions from Week 7
(`05_advanced_models.ipynb`) and:

- Computes metrics beyond R²: MAE, MAPE, and MdAPE (already used in Week 6/7, restated here as
  the primary deliverable).
- Breaks metrics down by `ClosePrice` band to see which price ranges the model handles well vs.
  poorly.
- Writes a summary table to `metrics_summary.csv`.


## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error


## 2. Load Week 7's test predictions

Reuses `y_true` / `y_pred` saved at the end of `05_advanced_models.ipynb`, so metrics here match
the tuned gradient boosting model exactly -- no retraining, no risk of the split drifting between
notebooks.

In [2]:
DATA_DIR = "data/processed"

preds_df = pd.read_csv(f"{DATA_DIR}/test_predictions.csv")
y_test = preds_df["y_true"]
y_pred = preds_df["y_pred"]

print(f"Loaded {len(preds_df)} test predictions")
preds_df.head()


Loaded 22644 test predictions


,y_true,y_pred
0,2275500.0,2.481505e+06
1,9000.0,4.000161e+04
2,849000.0,1.048039e+06
3,782450.0,7.500575e+05
4,10400.0,5.552569e+04


## 3. Metrics helper (R², MAE, MAPE, MdAPE)

In [3]:
def compute_metrics(y_true, y_pred):
    return {
        "n": len(y_true),
        "r2": r2_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "mape": mean_absolute_percentage_error(y_true, y_pred),
        "mdape": np.median(np.abs((y_true - y_pred) / y_true)),
    }


## 4. Overall test-set metrics

In [4]:
overall_metrics = compute_metrics(y_test, y_pred)
overall_metrics["band"] = "Overall"
print(overall_metrics)


{'n': 22644, 'r2': 0.8897642741883105, 'mae': 145869.4671388716, 'mape': 2.0453582501983307, 'mdape': np.float64(0.16632043777419303), 'band': 'Overall'}


## 5. Metrics by price band

Splits the test set into quartiles of `ClosePrice` (using `y_test` itself, since that *is*
`ClosePrice` for the test rows) and computes the same metrics within each band. This surfaces
whether the model is systematically better or worse at the low end vs. high end of the market --
a single overall R² can hide that kind of split.

In [5]:
preds_df["price_band"] = pd.qcut(
    y_test, q=4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]
)

band_rows = []
for band, group in preds_df.groupby("price_band", observed=True):
    m = compute_metrics(group["y_true"], group["y_pred"])
    m["band"] = str(band)
    band_rows.append(m)

band_metrics_df = pd.DataFrame(band_rows)[["band", "n", "r2", "mae", "mape", "mdape"]]
band_metrics_df


,band,n,r2,mae,mape,mdape
0,Q1 (lowest),5674,-51.363322,39728.350280,7.638680,4.158700
1,Q2,5664,0.092263,84647.255464,0.229922,0.122204
2,Q3,5652,-0.739368,117087.539311,0.135796,0.096222
3,Q4 (highest),5654,0.701384,342488.278612,0.159786,0.121721


## 6. Combine overall + per-band metrics, save summary

In [6]:
overall_row = pd.DataFrame([overall_metrics])[["band", "n", "r2", "mae", "mape", "mdape"]]
metrics_summary_df = pd.concat([overall_row, band_metrics_df], ignore_index=True)

metrics_summary_df.to_csv("metrics_summary.csv", index=False)
metrics_summary_df


,band,n,r2,mae,mape,mdape
0,Overall,22644,0.889764,145869.467139,2.045358,0.166320
1,Q1 (lowest),5674,-51.363322,39728.350280,7.638680,4.158700
2,Q2,5664,0.092263,84647.255464,0.229922,0.122204
3,Q3,5652,-0.739368,117087.539311,0.135796,0.096222
4,Q4 (highest),5654,0.701384,342488.278612,0.159786,0.121721


## 7. Insights summary

Quick, automated read on which price band the model struggles with most, based on MAPE (the
scale-independent error metric, so it's comparable across bands with very different price
levels).

In [7]:
best_band = band_metrics_df.loc[band_metrics_df["mape"].idxmin()]
worst_band = band_metrics_df.loc[band_metrics_df["mape"].idxmax()]

print(f"Best-performing price band by MAPE:  {best_band['band']} "
      f"(MAPE={best_band['mape']:.2%}, MdAPE={best_band['mdape']:.2%}, n={int(best_band['n'])})")
print(f"Worst-performing price band by MAPE: {worst_band['band']} "
      f"(MAPE={worst_band['mape']:.2%}, MdAPE={worst_band['mdape']:.2%}, n={int(worst_band['n'])})")
print()
print("Overall R²:", round(overall_metrics['r2'], 4),
      "| Overall MAPE:", f"{overall_metrics['mape']:.2%}",
      "| Overall MdAPE:", f"{overall_metrics['mdape']:.2%}")


Best-performing price band by MAPE:  Q3 (MAPE=13.58%, MdAPE=9.62%, n=5652)
Worst-performing price band by MAPE: Q1 (lowest) (MAPE=763.87%, MdAPE=415.87%, n=5674)

Overall R²: 0.8898 | Overall MAPE: 204.54% | Overall MdAPE: 16.63%
